# Дополнительный блокнот к главе 6
**Постройте свою первую LLM, глава 6: Руководство по выживанию с NumPy и PyTorch**

Запустите эти ячейки сверху вниз, чтобы увидеть основы работы с тензорами, маскирование, бродкастинг и небольшой цикл обучения.

**Маршрут:** импорт → численные примеры с тензорами → предварительное знакомство с вниманием и эмбеддингами (главы 10 и 9) → цикл обучения из пяти шагов и снижение потерь на примере `y = 2x + 1`.

In [ ]:
# ===== ИМПОРТЫ =====
# torch: Библиотека PyTorch для операций с тензорами (думайте: многомерные массивы)
# numpy: Классическая библиотека для численных вычислений (PyTorch вдохновлён ею)
# torch.nn: Строительные блоки нейронных сетей (слои и т.д.)
import torch, numpy as np
import torch.nn as nn

print('Версия Torch:', torch.__version__)
print('CUDA доступна:', torch.cuda.is_available())
# CUDA = вычисления на GPU. Если True, мы можем использовать ускорение GPU (в 10-100 раз быстрее для глубокого обучения)

## Что такое тензор?

**Тензор** (tensor) - это многомерный массив чисел:
- **1D тензор** = список `[1, 2, 3]` (как строка в электронной таблице)
- **2D тензор** = таблица/матрица (строки и столбцы)
- **3D тензор** = стопка таблиц (как несколько листов в Excel)

**Почему PyTorch вместо NumPy?**
Оба работают с многомерными массивами, но PyTorch добавляет:
1. **Поддержку GPU** - перемещение данных на GPU для ускорения в 10-100 раз
2. **Автоматические градиенты** - вычисляет производные для обучения нейронных сетей
3. **Слои нейронных сетей** - готовые строительные блоки

## Создание тензоров
Из данных Python/NumPy и с общими правилами заполнения.

In [ ]:
# ===== Создание тензоров =====

# Из данных Python
data = torch.tensor([[1, 2, 3], [4, 5, 6]])  # Тензор 2×3

# Правила заполнения (создание тензоров, заполненных определёнными значениями)
zeros = torch.zeros(3, 4)     # Тензор 3×4 из нулей
ones = torch.ones(2, 3, 4)    # Тензор 2×3×4 из единиц
uniform = torch.rand(3, 4)    # Случайные значения в [0, 1)
normal = torch.randn(3, 4)    # Случайные значения из нормального распределения (среднее=0, std=1)
integers = torch.randint(0, 10, (3, 4))  # Случайные целые числа в [0, 10)

# Последовательности с интервалом (как range в Python)
sequence = torch.arange(0, 10, 2)  # [0, 2, 4, 6, 8]
linspace = torch.linspace(0, 1, 5)  # 5 равномерно распределённых точек от 0 до 1

# Типы данных (dtype) - контроль точности
float_tensor = torch.tensor([1.0, 2.0], dtype=torch.float32)  # 32-битные числа с плавающей точкой (по умолчанию)
int_tensor = torch.tensor([1, 2], dtype=torch.long)           # 64-битные целые числа (для индексов)

# NumPy ↔ PyTorch (они разделяют память - изменения в одном влияют на другое!)
np_data = np.array([1, 2, 3], dtype=np.float32)
torch_from_np = torch.from_numpy(np_data)  # разделяет память с np_data
back_to_np = torch_from_np.numpy()         # обратно в NumPy

# Размещение на устройстве - CPU или GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_tensor = torch.randn(3, 4, device=device, dtype=torch.float32)

print('Форма zeros:', zeros.shape)
print('Устройство gpu_tensor:', gpu_tensor.device)

## Воспроизводимость: Установка seed
Критично для отладки и сравнения экспериментов

In [ ]:
# Фиксация случайного состояния для воспроизводимости
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Теперь эти значения будут идентичными при каждом запуске
a = torch.randn(3, 4)
b = torch.randn(3, 4)

# Попробуйте без seed - результаты меняются каждый раз
# Но с seed они воспроизводимы
torch.manual_seed(42)
x1 = torch.randn(2, 3)
torch.manual_seed(42)
x2 = torch.randn(2, 3)
print('С seed тензоры совпадают:', torch.allclose(x1, x2))

# Почему это важно: Воспроизводимые эксперименты для отладки и исследований

## От списков к тензорам: Построение размерностей
Связываем простые списки из главы 5 с многомерными тензорами

In [ ]:
# Шаг 1: 1D тензоры (напоминание из главы 5)
# Из главы 5: идентификаторы токенов от токенизатора
token_ids = [2, 3, 4, 6]
tokens_1d = torch.tensor(token_ids)
print(f"Форма 1D: {tokens_1d.shape}")  # torch.Size([4])
print(f"Данные: {tokens_1d}")

In [ ]:
# Шаг 2: Пакетирование (1D → 2D)
# Обработка нескольких предложений одновременно
batch = torch.tensor([
    [2, 3, 4, 6],    # предложение 1
    [5, 7, 8, 9]     # предложение 2
])
print(f"Форма 2D батча: {batch.shape}")  # torch.Size([2, 4])
print("Первое предложение:", batch[0])
print("Второй токен первого предложения:", batch[0, 1])

In [ ]:
# Шаг 3: Эмбеддинги (2D → 3D)
# Добавляем 768 чисел на токен (в стиле GPT-2)
embeddings_3d = torch.randn(2, 4, 768)
print(f"Форма 3D эмбеддингов: {embeddings_3d.shape}")  # torch.Size([2, 4, 768])

# Навигация по размерностям: батч → токен → признаки
print("Эмбеддинги первого предложения:", embeddings_3d[0].shape)      # (4, 768)
print("Первый токен первого предложения:", embeddings_3d[0, 0].shape)  # (768,)

In [ ]:
# Шаг 4: Предварительный просмотр голов внимания (3D → 4D)
# Многоголовое внимание добавляет ещё одно измерение (не беспокойтесь о деталях пока)
attention_4d = torch.randn(2, 8, 4, 4)  # (батч, головы, последовательность, последовательность)
print(f"Форма 4D внимания: {attention_4d.shape}")
print("Интерпретация формы: (батч, головы, длина_последовательности, длина_последовательности)")

## Изменение формы, сжатие, перестановка
Манипуляции с формой, которые вы будете использовать постоянно.

In [ ]:
# ===== Операции изменения формы =====
# Изменение формы = реорганизация данных без изменения значений (как перестановка коробок на складе)

x = torch.arange(12)               # Создаём 1D тензор [0,1,2,...,11], форма (12,)
print(f'Начало: {x.shape}')

# view() - изменяет форму, но требует "непрерывной" памяти (данные расположены последовательно)
x = x.view(3, 4)                   # Изменяем форму на (3, 4): 3 строки × 4 столбца
print(f'После view(3,4): {x.shape}')

# reshape() - как view(), но обрабатывает неконтинуальные тензоры (безопаснее, используйте при сомнении)
x = x.reshape(2, 2, 3)             # Изменяем форму на (2, 2, 3)
print(f'После reshape(2,2,3): {x.shape}')

# -1 означает "вычисли эту размерность за меня"
x = x.view(-1, 3)                  # -1 становится 4 (12 элементов всего ÷ 3 = 4)
print(f'После view(-1,3): {x.shape}')

# unsqueeze/squeeze - добавить или удалить размерности размера 1
y = torch.randn(3, 4)
y = y.unsqueeze(0)                 # (1, 3, 4) - добавляем размерность батча в позиции 0
y = y.unsqueeze(-1)                # (1, 3, 4, 1) - добавляем размерность в конце
y = y.squeeze()                    # Удаляем ВСЕ размерности размера 1 → (3, 4)
print(f'После squeeze: {y.shape}')

# permute - переупорядочить размерности (как транспонирование, но для любого числа измерений)
z = torch.randn(2, 3, 4)           # (батч, последовательность, признаки)
z = z.permute(0, 2, 1)             # (батч, признаки, последовательность) - меняем местами последние два измерения
print(f'После permute: {z.shape}')

# flatten - свернуть размерности
t = torch.randn(2, 3, 4)
t_flat = t.flatten()               # (24,) - все измерения свёрнуты
t_flat_features = t.flatten(1)     # (2, 12) - сворачиваем, начиная с измерения 1
print(f'Полностью свёрнуто: {t_flat.shape}')
print(f'Свёрнуты признаки: {t_flat_features.shape}')

## Индексация и срезы
Выбор батчей, токенов и использование булевых масок.

In [ ]:
# ===== Индексация и срезы =====
# Навигация по многомерным данным как по папкам: батч → токен → признаки

# Создаём 4D тензор, симулирующий внимание: (батч, головы, последовательность, последовательность)
x = torch.randn(2, 4, 6, 6)  # 2 батча, 4 головы, 6 токенов, 6 токенов

# Базовая индексация
first_batch = x[0]              # Форма: (4, 6, 6) - первый батч, все головы
first_head = x[0, 0]            # Форма: (6, 6) - первый батч, первая голова
single_value = x[0, 0, 0, 0]    # Форма: () - одно число

# Срезы с двоеточием
first_two_batches = x[:2]       # Форма: (2, 4, 6, 6) - первые 2 батча
all_but_last_token = x[:, :, :-1, :]  # Форма: (2, 4, 5, 6) - удаляем последний токен
every_other_head = x[:, ::2]    # Форма: (2, 2, 6, 6) - головы 0 и 2

# Отрицательные индексы считают с конца
last_token = x[:, :, -1, :]     # Форма: (2, 4, 6) - последний токен в каждой последовательности

# Булево маскирование (фильтрация по условию)
mask = torch.tensor([True, False, True, False])
filtered_heads = x[0, mask]     # Форма: (2, 6, 6) - только головы 0 и 2

print(f'Исходная форма: {x.shape}')
print(f'Форма первого батча: {first_batch.shape}')
print(f'Всё кроме последнего токена: {all_but_last_token.shape}')
print(f'Отфильтрованные головы: {filtered_heads.shape}')

## Внимание: Сердце трансформеров

**Общая картина:** Внимание вычисляет взвешенное среднее всех слов, где веса получаются из оценок релевантности. Как чтение "The cat sat on the mat" (Кот сел на коврик), при обработке "sat" (сел) вы смотрите назад на "cat" (кто сел?) и "mat" (куда сел?).

Мы построим это в 3 шага:
1. Базовая математика (4 операции)
2. Добавление причинного маскирования (предотвращение подглядывания в будущее)
3. Производственный ярлык (PyTorch делает всё это)

In [ ]:
import torch
import torch.nn.functional as F

# ===== Базовая математика внимания =====
# Внимание: Запрос (Query, что я ищу?), Ключ (Key, что у меня есть?), Значение (Value, что вернуть?)

# Симулируем эмбеддинги для 5 токенов в батче из 2
batch, seq_len, d_head = 2, 5, 64
Q = torch.randn(batch, seq_len, d_head)  # (2, 5, 64)
K = torch.randn(batch, seq_len, d_head)
V = torch.randn(batch, seq_len, d_head)

# Шаг 1: Вычисляем оценки (насколько каждый токен соответствует каждому другому?)
# Оператор @ - это матричное умножение (то же, что torch.matmul)
# K.transpose(-2, -1) меняет местами последние два измерения: (2, 5, 64) → (2, 64, 5)
# Результат: (2, 5, 64) @ (2, 64, 5) → (2, 5, 5) - сетка оценок 5×5 на батч
scores = Q @ K.transpose(-2, -1)
print(f'Форма оценок: {scores.shape}')  # (2, 5, 5)

# Шаг 2: Масштабирование (предотвращение насыщения softmax)
# Без масштабирования большие скалярные произведения → softmax даёт ~1 для максимума, ~0 для остальных
scores = scores / (d_head ** 0.5)  # делим на sqrt(64) = 8

# Шаг 3: Softmax (превращаем оценки в вероятности)
# dim=-1 означает "вдоль последней размерности" (по ключам)
attn_weights = torch.softmax(scores, dim=-1)  # каждая строка суммируется в 1

# Шаг 4: Взвешенная сумма (смешиваем значения, используя вероятности)
output = attn_weights @ V  # (2, 5, 64)

print(f'Форма выхода: {output.shape}')  # такая же, как вход: (2, 5, 64)
print(f'Веса внимания суммируются в 1: {attn_weights[0, 0].sum():.4f}')
print(f'\nВеса внимания токена 2: {attn_weights[0, 2]}')
print('(показывает, насколько токен 2 обращает внимание на каждый из 5 токенов)')

## Эмбеддинги токенов и позиций

Каждая LLM начинает с преобразования идентификаторов токенов в плотные векторы.

**Проблема:** Нейронные сети не могут обрабатывать сырой текст вроде "cat" (кот). Идентификаторы токенов (как `5` для "cat") произвольны - ID 5 не "ближе" к 6, чем к 500.

**Решение:** Сопоставляем каждый идентификатор токена с обучаемым вектором (768 чисел для GPT-2). Похожие слова обучаются похожим векторам в процессе тренировки.

In [ ]:
# ===== Эмбеддинги токенов и позиций =====
# Размерности GPT-2
vocab_size, d_model, max_seq_len = 50257, 768, 1024

# Эмбеддинги токенов: таблица поиска (как словарь: token_id → вектор)
# nn.Embedding создаёт таблицу с vocab_size строками и d_model столбцами
# Каждая строка - это 768-мерный вектор, представляющий один токен
token_embedding = nn.Embedding(vocab_size, d_model)

# Создаём случайные идентификаторы токенов (представьте, что они пришли от токенизатора)
token_ids = torch.randint(0, vocab_size, (2, 5))  # 2 предложения, по 5 токенов каждое

# Ищем эмбеддинги (просто индексация в таблице!)
token_vectors = token_embedding(token_ids)
print(f'Эмбеддинги токенов: {token_vectors.shape}')  # (2, 5, 768)

# Эмбеддинги позиций: где в последовательности (токен 0, токен 1 и т.д.)
# Та же идея: таблица поиска, где строка i представляет "позицию i"
pos_embedding = nn.Embedding(max_seq_len, d_model)
position_ids = torch.arange(5).unsqueeze(0).expand(2, -1)  # [[0,1,2,3,4], [0,1,2,3,4]]
pos_vectors = pos_embedding(position_ids)
print(f'Эмбеддинги позиций: {pos_vectors.shape}')  # (2, 5, 768)

# Объединяем: поэлементное сложение (та же позиция, та же форма!)
# Почему сложение, а не конкатенация? Сложение сохраняет размерность в 768 (не 1536)
# Модель учится кодировать И значение И позицию в одном векторе
input_embeddings = token_vectors + pos_vectors
print(f'Объединённые: {input_embeddings.shape}')  # (2, 5, 768)

# Подсчёт параметров (сколько чисел нужно обучить?)
token_params = vocab_size * d_model   # 50,257 токенов × 768 измерений
pos_params = max_seq_len * d_model    # 1,024 позиций × 768 измерений
print(f'Параметры токенов: {token_params:,}')      # 38,597,376
print(f'Параметры позиций: {pos_params:,}')        # 786,432
print(f'Всего: {token_params + pos_params:,}')     # 39,383,808

In [ ]:
# Обнуляем градиенты → прямой проход → потери → обратный проход → обновление весов.
# В LLM между четвёртым и пятым шагами ограничиваем норму градиента.
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)

class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.norm(x)
        x = self.fc2(x)
        return x


model = SimpleMLP(16, 32, 2)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
inputs = torch.randn(64, 16)
labels = torch.randint(0, 2, (64,))

optimizer.zero_grad(set_to_none=True)
logits = model(inputs)
loss = loss_fn(logits, labels)
print(f"до шага: потери={loss.item():.4f}")
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

with torch.no_grad():
    after = loss_fn(model(inputs), labels)
print(f"после шага: потери={after.item():.4f}")
print("Метки случайны. Точность на обучающих примерах не показывает способность к обобщению.")

### Как снижаются потери

Обучим модель с одним весом и одним смещением воспроизводить правило `y = 2x + 1`.

In [ ]:
import torch
torch.manual_seed(42)
x = torch.linspace(0, 1, 32).unsqueeze(1)
y = 2 * x + 1
model = torch.nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.5)
loss_fn = torch.nn.MSELoss()
for step in range(1, 51):
    opt.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    opt.step()
    if step % 10 == 0:
        print(f"шаг {step:2d}: потери = {loss.item():.6f}")
print(f"результат: y = {model.weight.item():.2f}x + {model.bias.item():.2f}")

In [ ]:
# Полный модуль эмбеддингов (предпросмотр класса из главы 9)
class GPT2Embeddings(nn.Module):
    """Объединяет эмбеддинги токенов + позиций как GPT-2 (полностью в главе 9)."""

    def __init__(self, vocab_size, max_seq_len, embed_dim):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        # ID позиций на ТОМ ЖЕ устройстве, что токены (классическое исправление сбоя)
        position_ids = torch.arange(seq_len, device=token_ids.device)
        token_emb = self.token_embed(token_ids)     # (batch, seq, embed_dim)
        pos_emb = self.pos_embed(position_ids)      # (seq, embed_dim), broadcasts
        return token_emb + pos_emb

# Тестируем
embed_layer = GPT2Embeddings(vocab_size=50257, max_seq_len=1024, embed_dim=768)
token_ids = torch.randint(0, 50257, (2, 10))
output = embed_layer(token_ids)
print(f"Выход эмбеддингов: {output.shape}")  # (2, 10, 768)

# Всего параметров
total_params = sum(p.numel() for p in embed_layer.parameters())
print(f"Всего параметров эмбеддингов: {total_params:,}")  # 39,383,808

Обратите внимание: в этом классе нет dropout. MiniGPT добавляет dropout после этой суммы при сборке полной модели в главе 11. Готовая версия из главы 9 также проверяет длину последовательности и инициализирует веса. Этот предпросмотр использует сигнатуру главы 9 (`embed_dim`, `token_embed`, `pos_embed`), чтобы код переносился без изменений.

## Цикл обучения

**Ключевые концепции:**
- `nn.Module`: Базовый класс для слоёв нейронной сети. Ваша модель наследуется от него.
- `super().__init__()`: Вызывает инициализацию родительского класса (обязательный шаблонный код)
- `nn.Linear(in, out)`: Слой матричного умножения (матрица весов in×out)
- `nn.ReLU()`: Функция активации - сохраняет положительные значения, обнуляет отрицательные
- `CrossEntropyLoss`: Измеряет, насколько неправильны предсказания классификации
- `optimizer.zero_grad()`: Очищает старые градиенты (они накапливаются по умолчанию!)
- `.backward()`: Вычисляет градиенты с помощью автоматического дифференцирования
- `.step()`: Обновляет веса, используя вычисленные градиенты

In [ ]:
# Шаг 2: Добавление причинного маскирования (предотвращение обмана)

# Проблема: Если токен 2 может видеть токены 3 и 4, он может обманывать во время обучения!
# Решение: Блокируем будущие позиции, устанавливая их оценки в -inf

# Как выглядит маска (False = разрешить, True = заблокировать):
# Токен 0 может видеть: [0]           ← только себя
# Токен 1 может видеть: [0, 1]        ← прошлое + себя
# Токен 2 может видеть: [0, 1, 2]     ← прошлое + себя
# Токен 3 может видеть: [0, 1, 2, 3]
# Токен 4 может видеть: [0, 1, 2, 3, 4]

seq_len = 5
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

# Применяем маску перед softmax
scores = Q @ K.transpose(-2, -1) / (d_head ** 0.5)
scores = scores.masked_fill(causal_mask, float('-inf'))  # -inf становится 0 после softmax
attn_weights = torch.softmax(scores, dim=-1)

print("Веса причинного внимания (токен 2 может видеть только токены 0,1,2):")
print(attn_weights[0, 2])  # позиции 3 и 4 равны нулю
print("\nТеперь модель учится предсказывать 'mat' (коврик), не видя сначала 'mat'!")

In [ ]:
# Шаг 3: Производственный ярлык (одна строка)

# Вы только что изучили ручной процесс из 4 шагов, чтобы понять, что происходит.
# На практике PyTorch делает всё это за вас:

output = F.scaled_dot_product_attention(
    Q, K, V,
    is_causal=True  # автоматически применяет причинное маскирование
)

print(f"Форма выхода: {output.shape}")  # (2, 5, 64)

# Почему использовать это вместо ручного?
# - В 2-4 раза быстрее (использует FlashAttention)
# - Меньше памяти (не хранит полную матрицу внимания)
# - Обрабатывает граничные случаи (числовая стабильность, dropout, бродкастинг масок)

print("\n✅ Когда использовать ручное vs. производственное:")
print("   Обучение: Пишите вручную, чтобы понять математику")
print("   Продакшн: Всегда используйте F.scaled_dot_product_attention")

In [ ]:
seq_len = 5
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
print(mask)

## Примеры бродкастинга
Добавление смещения и маскирование с бродкастингом.

In [ ]:
x = torch.randn(3, 4)
y = x + 5
batch = torch.randn(32, 10, 768)
bias = torch.randn(768)
result = batch + bias  # смещение расширяется
scores = torch.randn(4, 8, 10, 10)
mask = torch.triu(torch.ones(1, 1, 10, 10), 1)
masked = scores + mask * -1e9
print('форма результата:', result.shape)

## Основные операции: Поэлементные, Matmul, Редукция

**Поэлементные операции** работают позиция за позицией, как сложение двух электронных таблиц ячейка за ячейкой. Если `a` и `b` оба тензоры 3×4, то `a + b` складывает `a[0,0]` с `b[0,0]`, `a[0,1]` с `b[0,1]` и так далее. Одинаковая форма на входе, одинаковая форма на выходе.

In [ ]:
a = torch.randn(3, 4)
b = torch.randn(3, 4)
add = a + b
mul = a * b
square = a ** 2
exp = torch.exp(a)
x = torch.randn(32, 10, 64)
W = torch.randn(64, 128)
y = x @ W
total = a.sum()
row_sums = a.sum(dim=1, keepdim=True)
col_means = a.mean(dim=0)
max_vals, max_idx = a.max(dim=1)
c = torch.cat([a, b], dim=0)
d = torch.stack([a, b], dim=0)
logits = torch.randn(3, 5)
probs = torch.softmax(logits, dim=-1)
print('форма y:', y.shape)
print('суммы строк probs:', probs.sum(dim=-1))

## Autograd: Автоматические градиенты

**Что такое градиент?** Производная - насколько меняется выход при изменении входа?

**Почему это важно?** Обучение нейронной сети означает корректировку миллионов параметров. Градиенты говорят нам, в каком направлении корректировать. Autograd в PyTorch делает это автоматически.

In [ ]:
# Простой вычислительный граф: x → y → z
x = torch.tensor([2.0, 3.0], requires_grad=True)  # отслеживаем операции с x
y = x ** 2        # y = [4.0, 9.0]
z = y.sum()       # z = 13.0

# Вычисляем градиенты автоматически
z.backward()      # "как изменится z, если я изменю x?"
print('Градиенты:', x.grad)  # tensor([4., 6.])

# Что только что произошло?
# z = (x²).sum() → dz/dx = 2x
# При x=[2, 3], градиенты равны 2*[2, 3] = [4, 6]
# backward() вычислил это, пройдя по графу назад!

print('\n✅ Ручная проверка: dz/dx = 2x')
print(f'   При x=[2, 3]: 2*x = {2 * x.detach()}')

# Когда прекратить отслеживание (экономит память во время вывода):
print('\n--- Остановка отслеживания градиентов ---')

# Вариант 1: Контекстный менеджер (для блока кода)
with torch.no_grad():
    y_no_grad = x * 2  # нет отслеживания градиентов
    print(f'Градиенты не вычисляются: {y_no_grad}')

# Вариант 2: Отсоединение (для одного тензора)
detached = x.detach()  # новый тензор, нет истории градиентов
print(f'Отсоединённый тензор: {detached}')

## Резюме

Теперь вы увидели все основные операции PyTorch для разработки LLM:

✅ **Основы тензоров** - создание, типы данных, устройства
✅ **Воспроизводимость** - seed для отладки
✅ **Построение размерностей** - прогрессия 1D → 2D → 3D → 4D
✅ **Изменение формы и индексация** - навигация по многомерным данным
✅ **Механизм внимания** - ручной и производственный паттерны
✅ **Причинное маскирование** - предотвращение подглядывания будущих токенов
✅ **Эмбеддинги** - представления токенов и позиций
✅ **Бродкастинг** - автоматическое расширение формы
✅ **Математические операции** - поэлементные, matmul, редукция
✅ **Autograd** - автоматическое дифференцирование
✅ **Циклы обучения** - прямой проход, обратный проход, оптимизация с обрезкой градиентов

**Следующие шаги:** Глава 7 покажет вам, как подготовить реальные текстовые данные для обучения!